In [ ]:
#Create table, Only run manually first time then freeze cell
spark.sql("""
CREATE TABLE IF NOT EXISTS ViewDefinitionAudit (
    schema_name STRING,
    object_name STRING,
    object_type STRING,
    object_definition STRING,
    definition_hash STRING,
    capture_datetime TIMESTAMP
)
USING DELTA
""")

In [ ]:
from notebookutils import mssparkutils
from pyspark.sql import functions as F
from pyspark.sql.window import Window

token = mssparkutils.credentials.getToken(
    "https://database.windows.net/"
)

jdbc_url = (
    "jdbc:sqlserver:datawarehouse.fabric.microsoft.com;"
    "database=dev_database;"
    "encrypt=true;"
    "trustServerCertificate=false;"
)

sql_query = """
SELECT
    s.name AS schema_name,
    o.name AS object_name,
    'View' AS object_type,
    m.definition AS object_definition,
    HASHBYTES(
        'SHA2_256',
        CAST(ISNULL(m.definition, '') AS NVARCHAR(MAX))
    ) AS definition_hash
FROM sys.objects o
JOIN sys.schemas s
    ON o.schema_id = s.schema_id
LEFT JOIN sys.sql_modules m
    ON o.object_id = m.object_id
WHERE o.type = 'V'
"""

current_df = (
    spark.read
        .format("jdbc")
        .option("url", jdbc_url)
        .option("query", sql_query)
        .option("accessToken", token)
        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
        .load()
)


current_df = current_df.withColumn(
    "definition_hash",
    F.upper(F.hex("definition_hash"))
)

#display(current_df)

In [ ]:
audit_table = "ViewDefinitionAudit"

# Existing audit history
audit_df = spark.table(audit_table)

# Latest hash per view
w = Window.partitionBy(
    "schema_name",
    "object_name"
).orderBy(F.col("capture_datetime").desc())

latest_df = (
    audit_df
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .select(
        "schema_name",
        "object_name",
        F.col("definition_hash").alias("previous_hash")
    )
)

# Find new or changed views, Create df with records if previous hash is new or not equal to previous hash
changes_df = (
    current_df.alias("curr")
    .join(
        latest_df.alias("hist"),
        ["schema_name", "object_name"],
        "left"
    )
    .filter(
        F.col("previous_hash").isNull() |
        (F.col("definition_hash") != F.col("previous_hash"))
    )
    .select(
        "schema_name",
        "object_name",
        "object_type",
        "object_definition",
        "definition_hash"
    )
    .withColumn(
        "capture_datetime",
        F.current_timestamp()
    )
)

In [ ]:
change_count = changes_df.count()

if change_count > 0:
    (
        changes_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(audit_table)
    )

#print(f"{change_count} new/changed views inserted.")